# BM25 baseline model

## Imports

In [11]:
import os
import numpy as np
import pandas as pd
import json
import nltk
from nltk.stem import SnowballStemmer
from nltk.corpus import stopwords
import re
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

# only first time
# nltk.download('stopwords')

## Data loading

In [12]:
# path for the jurisprudence corpus
jurisprudence_path = 'jurisprudence_omgevingswet.jsonl'

# load JSONL into DataFrame
jur_data = []
with open(jurisprudence_path, 'r', encoding='utf-8') as f:
    for line in f:
        jur_data.append(json.loads(line))

df_jurisprudence = pd.DataFrame(jur_data)
print(f"Jurisprudence dataset shape: {df_jurisprudence.shape}")
print(f"Columns: {df_jurisprudence.columns.tolist()}")


# path for the legislation corpus
legislation_path = 'legislation_omgevingswet.jsonl'

# load JSONL into DataFrame
leg_data = []
with open(legislation_path, 'r', encoding='utf-8') as f:
    for line in f:
        leg_data.append(json.loads(line))

df_legislation = pd.DataFrame(leg_data)
print(f"Legislation dataset shape: {df_legislation.shape}")
print(f"Columns: {df_legislation.columns.tolist()}")


Jurisprudence dataset shape: (3906, 5)
Columns: ['ecli', 'title', 'updated', 'summary', 'full_text']
Legislation dataset shape: (725, 4)
Columns: ['source_type', 'id', 'title', 'text']


### Corpus creation

In [13]:
jurisprudence_docs = [
    Document(
        page_content=row['full_text'], 
        metadata={
            "ecli": row['ecli'], 
            "title": row['title'], 
            "type": "jurisprudence",
            "updated": row['updated']
        }
    )
    for _, row in df_jurisprudence.iterrows()
]

legislation_docs = [
    Document(
        page_content=row['text'], 
        metadata={
            "id": row['id'], 
            "title": row['title'], 
            "source": row['source_type'],
            "type": "legislation"
        }
    )
    for _, row in df_legislation.iterrows()
]

### preprocessing function

In [14]:
dutch_stopwords = set(stopwords.words('dutch'))
stemmer = SnowballStemmer("dutch")

def preprocess_func(text):
    # 1. Lowercasing
    text = text.lower()
    # 2. Tokenization (alleen woorden)
    tokens = re.findall(r'\w+', text)
    # 3. Stopword removal & 4. Stemming
    return [stemmer.stem(t) for t in tokens if t not in dutch_stopwords]

### Create two BM25 Retrievers
Because the Legislation data is shorter than the Jurisprudence data. There will be two seperate retrievers

In [15]:
legislation_retriever = BM25Retriever.from_documents(legislation_docs)
jurisprudence_retriever = BM25Retriever.from_documents(jurisprudence_docs)

# Definieer de parameters in een dictionary
leg_params = {"k1": 1.2, "b": 0.75}
jur_params = {"k1": 1.5, "b": 0.85}

# Geef ze mee tijdens de initialisatie
legislation_retriever = BM25Retriever.from_documents(
    legislation_docs, 
    preprocess_func=preprocess_func,
    bm25_params=leg_params
)
jurisprudence_retriever = BM25Retriever.from_documents(
    jurisprudence_docs, 
    preprocess_func=preprocess_func,
    bm25_params=jur_params
)

legislation_retriever.k = 3
jurisprudence_retriever.k = 3


### Query the Retriever

In [ ]:
# Test query
query = "vellen van houtopstand zonder vergunning monumentale boom achtertuin handhaving gemeente"

In [17]:
# Retrieve relevant documents
legis_results = legislation_retriever.invoke(query)
juris_results = jurisprudence_retriever.invoke(query)

# Display the results
print("Top Retrieved Documents Legislation:\n")
for i, doc in enumerate(legis_results, 1):
    print(f"{i}. {doc.page_content}\n")

print("Top Retrieved Documents Jurisprudence:\n")
for i, doc in enumerate(juris_results, 1):
    print(f"{i}. {doc.page_content}\n")



Top Retrieved Documents Legislation:

1. Artikel 4.35 (rijksregels houtopstanden) 1 De in artikel 4.3 bedoelde regels over het vellen en beheren van houtopstanden worden gesteld met het oog op de instandhouding van het bosareaal, de natuurbescherming of het beschermen van landschappelijke waarden. 2 De regels strekken er in ieder geval toe dat na het vellen of het anders tenietgaan van een houtopstand, herbeplanting plaatsvindt op bosbouwkundig verantwoorde wijze. 202031004-09-202008-07-202034985202311307-04-202305-04-202301-01-2024

2. Artikel 10.24 (bomen en beplantingen) Degene op wie een gedoogplicht op grond van paragraaf 10.3.2 rust, gedoogt dat de initiatiefnemer bomen en beplantingen rooit, inkort of snoeit voor zover die bomen en beplantingen hinderlijk zijn voor het tot stand brengen of opruimen van het werk van algemeen belang. 201615626-04-201623-03-20163396220238922-03-202320-03-202301-01-2024202017217-06-202012-02-202034986202311307-04-202305-04-202301-01-2024

3. Artikel